## Combine, Clean & Status — AL_MT / AL_ST 20-trial runs
Scans `RESULTS/` for both methods, reports failures/missing,
combines successful results into `COMBINED/`, applies quality filters,
and produces per-size MLPreprocessing visualizations.

In [1]:
import sys, os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
from joblib import Parallel, delayed

THERMOIFT_SRC = Path("../../thermoift/src").resolve()
if str(THERMOIFT_SRC) not in sys.path:
    sys.path.insert(0, str(THERMOIFT_SRC))

from thermoift import MLPreprocessing

In [2]:
AL_METHODS   = ["AL_MT", "AL_ST", "AL_V2_AL"]   # method subdirectories under RESULTS/ and COMBINED/
RESULTS_DIR  = Path("RESULTS")
COMBINED_DIR = Path("COMBINED")
IFT_SUBPATH  = "CSV/InterfacialProperties/feed_1_interfacial_results.csv"

N_JOBS = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 1))

# Quality filters (same as all other methods)
MAX_VAPOR_DENSITY  = 400    # kg/m3
MIN_IFT_THICKNESS  = 0.0    # nm
MAX_IFT_THICKNESS  = 7.5    # nm
MIN_GAMMA          = 0.05   # mN/m

COMPONENTS = [
    "carbon dioxide", "hydrogen", "argon", "nitrogen",
    "methane", "oxygen", "carbon monoxide", "hydrogen sulfide"
]
TARGET = "gamma"
Z_COLS = [f"z_{c}" for c in COMPONENTS]

print(f"N_JOBS   = {N_JOBS}")
print(f"Methods  : {AL_METHODS}")

N_JOBS   = 24
Methods  : ['AL_MT', 'AL_ST', 'AL_V2_AL']


In [3]:
def _process_trial(method, size, trial, task_map, combined_dir, ift_subpath,
                   min_ift, max_ift, min_gamma, max_rhov):
    import pandas as pd
    from pathlib import Path

    out_dir  = Path(combined_dir) / method / size
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{trial}.csv"

    dfs = []
    for task_id, folder in sorted(task_map.items()):
        try:
            df = pd.read_csv(Path(folder) / ift_subpath)
            df.insert(0, "task_id", task_id)
            dfs.append(df)
        except Exception as e:
            print(f"  Warning: {Path(folder).name}: {e}", flush=True)

    if not dfs:
        return None

    raw = pd.concat(dfs, ignore_index=True)

    mask_nan        = raw["gamma"].isna() | raw["interfacial_thickness"].isna()
    mask_thick_low  = raw["interfacial_thickness"] <= min_ift
    mask_thick_high = raw["interfacial_thickness"] > max_ift
    mask_gamma_low  = raw["gamma"] < min_gamma
    mask_rhoV       = raw["vapor_density"] >= max_rhov

    valid        = ~mask_nan
    n_nan        = int(mask_nan.sum())
    n_thick_low  = int((valid & mask_thick_low).sum())
    n_thick_high = int((valid & ~mask_thick_low & mask_thick_high).sum())
    n_gamma_low  = int((valid & ~mask_thick_low & ~mask_thick_high & mask_gamma_low).sum())
    n_rhoV       = int((valid & ~mask_thick_low & ~mask_thick_high & ~mask_gamma_low & mask_rhoV).sum())
    n_raw        = len(raw)

    clean   = raw[valid & ~mask_thick_low & ~mask_thick_high & ~mask_gamma_low & ~mask_rhoV]
    n_clean = len(clean)
    clean.to_csv(out_path, index=False)

    return {
        "method":        method,
        "size":          size,
        "trial":         trial,
        "tasks":         len(task_map),
        "raw_rows":      n_raw,
        "nan_gamma":     n_nan,
        "bad_thick_low": n_thick_low,
        "bad_thick_hi":  n_thick_high,
        "bad_gamma_low": n_gamma_low,
        "bad_rhoV":      n_rhoV,
        "clean_rows":    n_clean,
        "pct_clean":     round(100 * n_clean / n_raw, 1) if n_raw else 0.0,
    }


def _plot_one(method, size, combined_dir_str, thermoift_src_str, z_cols, target):
    import sys, matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import pandas as pd
    from pathlib import Path

    thermoift_src = Path(thermoift_src_str)
    if str(thermoift_src) not in sys.path:
        sys.path.insert(0, str(thermoift_src))
    from thermoift import MLPreprocessing, PLOT_SETTINGS as ps

    combined_dir = Path(combined_dir_str)
    csvs = sorted((combined_dir / method / size).glob("trial_*.csv"))
    if not csvs:
        return f"{method}/{size}: no combined CSVs — skipping"

    df = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
    out_folder = (combined_dir / method / size / "PLOTS").resolve()
    out_folder.mkdir(parents=True, exist_ok=True)
    folder_str = str(out_folder)

    z    = [c for c in z_cols if c in df.columns]
    prep = MLPreprocessing(df=df, features=["temperature", "pressure"] + z, target=target)

    prep.plot_scatter("temperature",        target, "pressure",     save_path="gamma_vs_T",         folder=folder_str); plt.close("all")
    prep.plot_scatter("pressure",           target, "temperature",  save_path="gamma_vs_P",         folder=folder_str); plt.close("all")
    prep.plot_scatter("liquid_density",     target, "temperature",  save_path="gamma_vs_rhoL",      folder=folder_str); plt.close("all")
    prep.plot_scatter("vapor_density",      target, "temperature",  save_path="gamma_vs_rhoV",      folder=folder_str); plt.close("all")
    prep.plot_scatter("interfacial_thickness", target, "temperature", save_path="gamma_vs_thickness", folder=folder_str); plt.close("all")

    fig, _ = prep.plot_histogram(target, bins=50, save_path=None)
    ps.save_plot(fig, "gamma_distribution",   folder=folder_str); plt.close("all")

    fig, _ = prep.plot_phase_envelope(group_by="temperature", value_col=target, save_path=None)
    ps.save_plot(fig, "gamma_phase_envelope", folder=folder_str); plt.close("all")

    return f"{method}/{size}: {len(df):,} rows from {len(csvs)} trials"


print("Helpers defined.")

Helpers defined.


### Discover expected (method, size, trial) combinations from OUTPUT/

In [4]:
# expected[(method, size, trial)] = n_compositions
expected = {}

for method in AL_METHODS:
    output_dir = Path(method) / "OUTPUT"
    if not output_dir.exists():
        print(f"  {method}/OUTPUT not found — skipping")
        continue
    for size_dir in sorted(output_dir.iterdir()):
        if not size_dir.is_dir():
            continue
        for csv in sorted(size_dir.glob("trial_*.csv")):
            trial = csv.stem
            n     = sum(1 for _ in csv.open()) - 1
            expected[(method, size_dir.name, trial)] = n

print(f"Expected: {len(expected)} (method, size, trial) combinations")
for method in AL_METHODS:
    keys   = [(m, s, t) for m, s, t in expected if m == method]
    sizes  = sorted({s for _, s, _ in keys})
    trials = sorted({t for _, _, t in keys})
    if keys:
        ex_n = expected[keys[0]]
        print(f"  {method}: {len(sizes)} sizes × {len(trials)} trials  "
              f"({ex_n} compositions per trial)")

Expected: 240 (method, size, trial) combinations
  AL_MT: 4 sizes × 20 trials  (25 compositions per trial)
  AL_ST: 4 sizes × 20 trials  (25 compositions per trial)
  AL_V2_AL: 4 sizes × 20 trials  (25 compositions per trial)


### Scan RESULTS/ and classify tasks

In [5]:
failures  = {}
missing   = {}
successes = {}

for (method, size, trial), n_expected in sorted(expected.items()):
    trial_dir = RESULTS_DIR / method / size / trial

    if not trial_dir.exists():
        missing[(method, size, trial)] = list(range(n_expected))
        continue

    by_task = defaultdict(list)
    for folder in trial_dir.iterdir():
        if not folder.is_dir():
            continue
        parts = folder.name.rsplit("_", 1)
        if len(parts) == 2 and parts[1].isdigit():
            by_task[int(parts[1])].append(folder)

    trial_failures  = []
    trial_missing   = []
    trial_successes = {}

    for task_id in range(n_expected):
        folders = by_task.get(task_id, [])
        if not folders:
            trial_missing.append(task_id)
        else:
            ok = [f for f in folders if (f / IFT_SUBPATH).exists()]
            if not ok:
                trial_failures.append(task_id)
            else:
                best = sorted(ok, key=lambda f: int(f.name.rsplit("_", 1)[0]))[-1]
                trial_successes[task_id] = best

    if trial_failures:
        failures[(method, size, trial)] = trial_failures
    if trial_missing:
        missing[(method, size, trial)] = trial_missing
    if trial_successes:
        successes[(method, size, trial)] = trial_successes

n_success = sum(len(v) for v in successes.values())
n_fail    = sum(len(v) for v in failures.values())
n_miss    = sum(len(v) for v in missing.values())
print(f"Successful tasks : {n_success}")
print(f"Failed tasks     : {n_fail}   (ran but produced no output)")
print(f"Missing tasks    : {n_miss}  (never ran)")

Successful tasks : 15000
Failed tasks     : 0   (ran but produced no output)
Missing tasks    : 0  (never ran)


### What is missing / failed

In [6]:
if not failures and not missing:
    print("All tasks completed successfully — nothing missing.")
else:
    if missing:
        print("=== Never ran (no result folder) ===")
        for (method, size, trial), task_ids in sorted(missing.items()):
            if len(task_ids) == expected[(method, size, trial)]:
                print(f"  {method}/{size}/{trial}  →  entire trial not submitted yet")
            else:
                print(f"  {method}/{size}/{trial}  →  {len(task_ids)} tasks never ran: {task_ids}")

    if failures:
        print("\n=== Ran but failed (no CSV output) ===")
        for (method, size, trial), task_ids in sorted(failures.items()):
            print(f"  {method}/{size}/{trial}  →  {len(task_ids)} failed tasks: {task_ids}")

All tasks completed successfully — nothing missing.


### Combine and clean — parallel across (method, size, trial) groups

In [7]:
COMBINED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Combining {len(successes)} groups with {N_JOBS} parallel workers...")

results = Parallel(n_jobs=N_JOBS, backend="multiprocessing")(
    delayed(_process_trial)(
        method, size, trial, task_map,
        COMBINED_DIR, IFT_SUBPATH,
        MIN_IFT_THICKNESS, MAX_IFT_THICKNESS, MIN_GAMMA, MAX_VAPOR_DENSITY
    )
    for (method, size, trial), task_map in sorted(successes.items())
)

trial_stats = [r for r in results if r is not None]
stats_df    = pd.DataFrame(trial_stats).sort_values(["method", "size", "trial"]).reset_index(drop=True)
print(f"Written {len(trial_stats)} combined CSVs to {COMBINED_DIR}/")

Combining 240 groups with 24 parallel workers...
Written 240 combined CSVs to COMBINED/


### Per-trial cleaning statistics

In [8]:
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 140)
print(stats_df.to_string(index=False))

  method size    trial  tasks  raw_rows  nan_gamma  bad_thick_low  bad_thick_hi  bad_gamma_low  bad_rhoV  clean_rows  pct_clean
   AL_MT N025 trial_00     25      4920         43              5             0             15         0        4857       98.7
   AL_MT N025 trial_01     25      4910         43              5             0             15         0        4847       98.7
   AL_MT N025 trial_02     25      4910         51              5             0             15         0        4839       98.6
   AL_MT N025 trial_03     25      4916         41              4             0             12         0        4859       98.8
   AL_MT N025 trial_04     25      4930         47              5             0             15         0        4863       98.6
   AL_MT N025 trial_05     25      4920         50              5             0             15         0        4850       98.6
   AL_MT N025 trial_06     25      4890         43              5             0             15         0

### Per-(method, size) cleaning statistics

In [9]:
size_summary = (
    stats_df
    .groupby(["method", "size"], sort=True)
    .agg(
        trials        = ("trial",         "count"),
        tasks_ok      = ("tasks",         "sum"),
        raw_rows      = ("raw_rows",      "sum"),
        nan_gamma     = ("nan_gamma",     "sum"),
        bad_thick_low = ("bad_thick_low", "sum"),
        bad_thick_hi  = ("bad_thick_hi",  "sum"),
        bad_gamma_low = ("bad_gamma_low", "sum"),
        bad_rhoV      = ("bad_rhoV",      "sum"),
        clean_rows    = ("clean_rows",    "sum"),
    )
)
size_summary["pct_clean"] = (100 * size_summary["clean_rows"] / size_summary["raw_rows"]).round(1)
print(size_summary.to_string())

tot = stats_df[['raw_rows','nan_gamma','bad_thick_low','bad_thick_hi',
                'bad_gamma_low','bad_rhoV','clean_rows']].sum()
print(f"\nTOTAL (all methods + sizes)")
print(f"  Raw rows          : {tot['raw_rows']:>8,}")
print(f"  NaN gamma dropped : {tot['nan_gamma']:>8,}")
print(f"  thick ≤ 0 dropped : {tot['bad_thick_low']:>8,}")
print(f"  thick > 7.5 nm    : {tot['bad_thick_hi']:>8,}")
print(f"  gamma < 0.05 mN/m : {tot['bad_gamma_low']:>8,}")
print(f"  rhoV ≥ 400 kg/m³  : {tot['bad_rhoV']:>8,}")
print(f"  Clean rows        : {tot['clean_rows']:>8,}  "
      f"({100*tot['clean_rows']/tot['raw_rows']:.2f}%)")

               trials  tasks_ok  raw_rows  nan_gamma  bad_thick_low  bad_thick_hi  bad_gamma_low  bad_rhoV  clean_rows  pct_clean
method   size                                                                                                                    
AL_MT    N025      20       500     98276        894             98             1            287         0       96996       98.7
         N050      20      1000    196368       1988            106             1            317         0      193956       98.8
         N075      20      1500    295145       3218            119             1            331         0      291476       98.8
         N100      20      2000    394174       4660            153             3            394         0      388964       98.7
AL_ST    N025      20       500     98706       1041             32             0             84         0       97549       98.8
         N050      20      1000    197655       2277            124             2         

### Task completion summary

In [10]:
print(f"{'Method':<8} {'Size':<6} {'Trials done':>12} {'Tasks OK':>10} {'Failed':>8} {'Missing':>9}")
print("-" * 60)

for method in AL_METHODS:
    sizes = sorted({s for m, s, _ in expected if m == method})
    for size in sizes:
        all_trials    = [(m, s, t) for m, s, t in expected if m == method and s == size]
        trials_done   = sum(1 for k in all_trials if k in successes)
        tasks_ok      = sum(len(v) for k, v in successes.items() if k[0] == method and k[1] == size)
        tasks_failed  = sum(len(v) for k, v in failures.items()  if k[0] == method and k[1] == size)
        tasks_missing = sum(len(v) for k, v in missing.items()   if k[0] == method and k[1] == size)
        print(f"{method:<8} {size:<6} {trials_done:>7}/{len(all_trials):<4} "
              f"{tasks_ok:>10} {tasks_failed:>8} {tasks_missing:>9}")

Method   Size    Trials done   Tasks OK   Failed   Missing
------------------------------------------------------------
AL_MT    N025        20/20          500        0         0
AL_MT    N050        20/20         1000        0         0
AL_MT    N075        20/20         1500        0         0
AL_MT    N100        20/20         2000        0         0
AL_ST    N025        20/20          500        0         0
AL_ST    N050        20/20         1000        0         0
AL_ST    N075        20/20         1500        0         0
AL_ST    N100        20/20         2000        0         0
AL_V2_AL N025        20/20          500        0         0
AL_V2_AL N050        20/20         1000        0         0
AL_V2_AL N075        20/20         1500        0         0
AL_V2_AL N100        20/20         2000        0         0


### Generate RERUN_trials.sh — resubmit failed and missing tasks

In [11]:
# Merge failures and missing into one rerun dict
rerun = {}   # (method, size, trial) → sorted list of task_ids
for key, ids in failures.items():
    rerun.setdefault(key, set()).update(ids)
for key, ids in missing.items():
    rerun.setdefault(key, set()).update(ids)
rerun = {k: sorted(v) for k, v in sorted(rerun.items())}

RERUN_PATH = Path("RERUN_trials.sh")

if not rerun:
    print("Nothing to rerun — all tasks succeeded.")
    if RERUN_PATH.exists():
        RERUN_PATH.unlink()
        print(f"Removed stale {RERUN_PATH}")
else:
    lines = [
        "#!/bin/bash",
        "# Auto-generated by clean_trials.ipynb — resubmit failed/missing tasks.",
        "# Run from within ActiveLearning/: bash RERUN_trials.sh",
        "",
        "set -euo pipefail",
        "",
        'SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"',
        'TRIAL_SCRIPT="${SCRIPT_DIR}/RUN_trial.sh"',
        "",
        'echo "Resubmitting failed/missing AL trial tasks..."',
        "",
    ]

    n_jobs  = 0
    n_tasks = 0
    for (method, size, trial), task_ids in rerun.items():
        array_str = ",".join(str(t) for t in task_ids)
        lines.append(
            f'sbatch --array={array_str} --chdir="${{SCRIPT_DIR}}" '
            f'"${{TRIAL_SCRIPT}}" {method} {size} {trial}'
        )
        n_jobs  += 1
        n_tasks += len(task_ids)

    lines += [
        "",
        f'echo ""',
        f'echo "Submitted {n_jobs} rerun jobs ({n_tasks} tasks total)."',
        'echo "Run clean_trials.ipynb again after jobs complete to verify."',
    ]

    RERUN_PATH.write_text("\n".join(lines) + "\n")
    RERUN_PATH.chmod(0o775)

    print(f"Written: {RERUN_PATH}  ({n_jobs} jobs, {n_tasks} tasks)")
    print()
    for (method, size, trial), task_ids in rerun.items():
        print(f"  {method}/{size}/{trial}  →  tasks {task_ids}")

Nothing to rerun — all tasks succeeded.
Removed stale RERUN_trials.sh


### MLPreprocessing plots — per (method, size)
Loads all clean trial CSVs and runs the 7-plot suite.
Saved to `COMBINED/{METHOD}/{SIZE}/PLOTS/`.

In [12]:
plot_jobs = [
    (method, size)
    for method in AL_METHODS
    for size in sorted({s for m, s, _ in expected if m == method})
]

print(f"Generating plots for {len(plot_jobs)} (method, size) combinations "
      f"with {min(N_JOBS, len(plot_jobs))} workers...")

msgs = Parallel(n_jobs=min(N_JOBS, len(plot_jobs)), backend="multiprocessing")(
    delayed(_plot_one)(method, size, str(COMBINED_DIR), str(THERMOIFT_SRC), Z_COLS, TARGET)
    for method, size in plot_jobs
)
for m in msgs:
    print(m)

Generating plots for 12 (method, size) combinations with 12 workers...
AL_MT/N025: 96,996 rows from 20 trials
AL_MT/N050: 193,956 rows from 20 trials
AL_MT/N075: 291,476 rows from 20 trials
AL_MT/N100: 388,964 rows from 20 trials
AL_ST/N025: 97,549 rows from 20 trials
AL_ST/N050: 194,975 rows from 20 trials
AL_ST/N075: 292,547 rows from 20 trials
AL_ST/N100: 389,975 rows from 20 trials
AL_V2_AL/N025: 97,045 rows from 20 trials
AL_V2_AL/N050: 194,524 rows from 20 trials
AL_V2_AL/N075: 292,287 rows from 20 trials
AL_V2_AL/N100: 389,909 rows from 20 trials
